<center>

# TP3 : Prompt Engineering
## Tâches de classification : Analyse de sentiment

</center>

**Objectifs**
- Présenter les LLM comme outil efficace pour les problèmes de classification.
- Illustrer le prompt engineering pour l'analyse de sentiment.
- Construire et évaluer des LLM sur des tâches de classification plus larges.

# 0. Préparer l'environnement

1. Dans VS Code, créer un dossier **SMA_TokPr** et ouvrir le terminal.
2. Installer **uv** (puis fermer et rouvrir VS Code) :
   - Windows (PowerShell) : `powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"`
   - macOS / Linux : `curl -LsSf https://astral.sh/uv/install.sh | sh`
3. Initialiser le projet : `uv init` (génère `pyproject.toml`).
4. Créer l'environnement virtuel : `uv venv`
5. Activer l'environnement (PowerShell) :
   - `Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process`
   - `.\.venv\Scripts\activate`
6. Installer toutes les dépendances d'un coup :
   `uv add ipykernel python-dotenv tiktoken langchain langchain-openai langchain-groq langchain-google-genai langchain-ollama pandas scikit-learn pydantic`
7. Créer un fichier **`.env`** à la racine du projet (même dossier que le notebook) :
```
GROQ_API_KEY=votre_cle_groq
TAVILY_API_KEY=votre_cle_tavily
GOOGLE_API_KEY=votre_cle_google
```
8. Ouvrir ce notebook, choisir le kernel `.venv` (Python), puis tester avec `print("Bonjour")`.

In [1]:
print("Bonjour")

Bonjour


In [2]:
import os
from dotenv import load_dotenv   # CORRECTION : et non `from dotenv.ipython import load_dotenv`

load_dotenv(override=True)

# Vérification (on n'affiche jamais la clé elle-même)
for k in ["GROQ_API_KEY", "GOOGLE_API_KEY", "TAVILY_API_KEY", "OPENAI_API_KEY"]:
    print(f"{k:16} ->", "OK" if os.getenv(k) else "absente")

GROQ_API_KEY     -> OK
GOOGLE_API_KEY   -> OK
TAVILY_API_KEY   -> OK
OPENAI_API_KEY   -> absente


# 1. Tokenisation avec tiktoken

In [3]:
import tiktoken

In [4]:
tokenizer = tiktoken.encoding_for_model("gpt-4o")
print(tokenizer.name)   # o200k_base

o200k_base


In [5]:
tokenizer = tiktoken.get_encoding("o200k_base")

In [6]:
prompt = "Vous êtes un expert en Intelligence artificielle. C'est quoi un machine learning ? "

In [7]:
tokens = tokenizer.encode(prompt)
print(tokens)
print("Nombre de tokens :", len(tokens))

[29038, 34077, 537, 8333, 469, 42378, 105453, 22380, 13, 363, 6616, 33399, 537, 7342, 7524, 1423, 220]
Nombre de tokens : 17


In [8]:
# CORRECTION : errors="replace" évite UnicodeDecodeError quand un caractère accentué
# est découpé sur plusieurs tokens
for token in tokens:
    t = tokenizer.decode_single_token_bytes(token).decode("utf-8", errors="replace")
    print(t, end="|")

Vous| êtes| un| expert| en| Intelligence| artific|ielle|.| C|'est| quoi| un| machine| learning| ?| |

# 2. Utilisation d'un LLM avec LangChain

On utilise **Groq** (gratuit, rapide) via la clé `GROQ_API_KEY` du fichier `.env`.
(Le code OpenAI du TP original est conservé en commentaire : il nécessite une clé `OPENAI_API_KEY` payante.)

In [9]:
from langchain_groq import ChatGroq
from IPython.display import Markdown, display

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# --- Version OpenAI du TP original (si vous avez une clé OPENAI_API_KEY) ---
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [11]:
import os, requests

r = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.getenv('GROQ_API_KEY')}"},
    timeout=30,
)
print(sorted(m["id"] for m in r.json()["data"]))

['allam-2-7b', 'canopylabs/orpheus-arabic-saudi', 'canopylabs/orpheus-v1-english', 'meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'openai/gpt-oss-safeguard-20b', 'qwen/qwen3.8-27b', 'whisper-large-v3', 'whisper-large-v3-turbo']


In [12]:
from langchain_groq import ChatGroq
from IPython.display import Markdown, display

MODEL = "openai/gpt-oss-120b"   # si 404, choisis un id affiché par la cellule 1
llm = ChatGroq(model=MODEL, temperature=0)

In [13]:
resp = llm.invoke([
    {"role": "system", "content": "Vous êtes un expert en intelligence artificielle"},
    {"role": "user",   "content": "C'est quoi un machine learning ?"},
])
display(Markdown(resp.content))

**Le machine learning (apprentissage automatique)** est une branche de l’intelligence artificielle (IA) qui permet à un ordinateur d’apprendre à réaliser une tâche à partir de données, sans être explicitement programmé pour chaque situation.

---

## 1. Principe de base
1. **Données d’entraînement** : on fournit à l’algorithme un grand nombre d’exemples (images, textes, mesures, etc.) accompagnés de la réponse attendue (étiquette, valeur à prédire, etc.).
2. **Modèle** : l’algorithme construit un modèle mathématique qui capture les relations entre les entrées et les sorties.
3. **Apprentissage** : le modèle ajuste ses paramètres (poids, coefficients…) pour minimiser l’erreur entre ses prédictions et les réponses réelles.
4. **Évaluation** : on teste le modèle sur des données jamais vues pour vérifier sa capacité à généraliser.
5. **Utilisation** : le modèle « déployé » peut alors faire des prédictions ou prendre des décisions sur de nouvelles données.

---

## 2. Types d’apprentissage

| Type | Description | Exemple d’application |
|------|-------------|-----------------------|
| **Apprentissage supervisé** | Le modèle apprend à partir d’exemples étiquetés (entrée → sortie). | Classification d’images (chat vs chien), prévision de ventes. |
| **Apprentissage non supervisé** | Le modèle découvre des structures cachées dans des données non étiquetées. | Segmentation de clients, réduction de dimension (PCA), clustering. |
| **Apprentissage semi‑supervisé** | Combine un petit jeu d’exemples étiquetés avec beaucoup d’exemples non étiquetés. | Reconnaissance de texte avec peu d’annotations. |
| **Apprentissage par renforcement** | Un agent apprend en interagissant avec un environnement et en recevant des récompenses/punitions. | Jeux vidéo (AlphaGo), robots qui apprennent à marcher. |
| **Apprentissage auto‑supervisé** | Le modèle crée ses propres étiquettes à partir des données (ex. prédire la partie masquée d’une image). | Modèles de langage comme GPT, pré‑entraînement d’images. |

---

## 3. Étapes typiques d’un projet de machine learning

1. **Collecte & nettoyage des données**  
   - Rassembler les sources, gérer les valeurs manquantes, normaliser les formats.
2. **Exploration & visualisation**  
   - Statistiques descriptives, graphiques, détection d’anomalies.
3. **Pré‑traitement**  
   - Encodage des variables catégorielles, mise à l’échelle, création de nouvelles caractéristiques (feature engineering).
4. **Choix du modèle**  
   - Régression linéaire, forêts aléatoires, réseaux de neurones, SVM, etc.
5. **Entraînement**  
   - Optimisation des paramètres (gradient descent, arbres de décision, etc.).
6. **Validation**  
   - Validation croisée, réglage d’hyper‑paramètres (grid search, Bayesian optimisation).
7. **Évaluation**  
   - Métriques : précision, rappel, F1‑score, AUC, RMSE, etc.
8. **Déploiement**  
   - API, micro‑services, edge devices, etc.
9. **Surveillance & mise à jour**  
   - Détecter la dérive de données, ré‑entraîner le modèle si nécessaire.

---

## 4. Exemples concrets

- **Reconnaissance d’image** : un réseau de neurones convolutif (CNN) apprend à identifier des objets (voitures, panneaux de signalisation) à partir de millions de photos.
- **Filtrage de spam** : un classifieur (ex. Naïve Bayes, SVM) apprend à distinguer les e‑mails légitimes des spams grâce à des mots‑clés et des métadonnées.
- **Prédiction de la demande** : un modèle de régression (ou LSTM pour séries temporelles) prédit le volume de ventes d’un produit à partir de l’historique, des saisons, des promotions, etc.
- **Assistants vocaux** : les modèles de langage (transformers) apprennent à comprendre et générer du texte à partir de gigantesques corpus de dialogues.

---

## 5. Pourquoi le machine learning est‑il important ?

- **Automatisation de tâches complexes** où les règles explicites sont difficiles à formuler.
- **Adaptabilité** : le modèle s’améliore avec de nouvelles données.
- **Scalabilité** : une fois entraîné, le modèle peut traiter des millions d’exemples en quelques millisecondes.
- **Découverte de connaissances** : il peut révéler des patterns invisibles à l’œil humain (ex. détection de fraudes, biomarqueurs médicaux).

---

## 6. Limites & bonnes pratiques

| Limite | Conséquence | Bonne pratique |
|--------|-------------|----------------|
| **Qualité des données** | Un modèle apprend les biais et les erreurs présentes dans les données. | Nettoyer, équilibrer les jeux de données, vérifier la représentativité. |
| **Sur‑apprentissage (overfitting)** | Le modèle performe bien sur les données d’entraînement mais échoue sur de nouvelles données. | Utiliser la validation croisée, régularisation, early stopping. |
| **Interprétabilité** | Certains modèles (ex. réseaux profonds) sont des « boîtes noires ». | Privilégier des modèles explicables (arbres de décision) ou appliquer des techniques d’explicabilité (SHAP, LIME). |
| **Coût computationnel** | Entraîner de gros modèles peut être très gourmand en ressources. | Optimiser le code, exploiter le GPU/TPU, choisir des architectures plus légères si possible. |
| **Éthique & biais** | Risque de discrimination ou de décisions injustes. | Auditer les biais, appliquer des métriques d’équité, impliquer des experts du domaine. |

---

### En résumé
Le **machine learning** consiste à faire « apprendre » à un ordinateur à partir de données, afin qu’il puisse réaliser des prédictions ou prendre des décisions sans programmation explicite pour chaque cas. Il repose sur des algorithmes, des modèles mathématiques et un processus itératif d’entraînement, de validation et de déploiement. C’est aujourd’hui un pilier central de nombreuses innovations technologiques, de la vision par ordinateur aux assistants conversationnels, en passant par la médecine personnalisée et la finance.

# 3. Analyse de sentiment : prompting zero-shot

In [14]:
reviews = [
    "Ce produit est incroyable, je le recommande vivement !",
    "Livraison en retard et article cassé. Très déçu.",
    "Il fait le travail, sans plus.",
]

system_zero = (
    "Tu es un classifieur de sentiment. Classe le texte en UNE seule étiquette parmi : "
    "positif, négatif, neutre. Réponds uniquement par l'étiquette, en minuscules, sans ponctuation."
)

LABELS = ["positif", "négatif", "neutre"]

def clean(text):
    t = (text or "").strip().lower().rstrip(".!")
    for lab in LABELS:
        if lab in t:          # garde l'étiquette même si le modèle ajoute des mots
            return lab
    return t or "(vide)"

for r in reviews:
    out = llm.invoke([{"role": "system", "content": system_zero},
                      {"role": "user", "content": r}])
    print(f"{clean(out.content):10} <- {r}")

positif    <- Ce produit est incroyable, je le recommande vivement !
negatif    <- Livraison en retard et article cassé. Très déçu.
neutre     <- Il fait le travail, sans plus.


# 4. Few-shot prompting (avec exemples)

In [15]:
system_few = system_zero + """

Exemples :
Texte : "Super qualité, je rachèterai sans hésiter." -> positif
Texte : "Service client nul, je me suis fait arnaquer." -> négatif
Texte : "Le colis est arrivé mardi." -> neutre
"""

for r in reviews:
    out = llm.invoke([{"role": "system", "content": system_few},
                      {"role": "user", "content": f'Texte : "{r}"'}])
    print(f"{out.content.strip():10} <- {r}")

positif    <- Ce produit est incroyable, je le recommande vivement !
negatif    <- Livraison en retard et article cassé. Très déçu.
neutre     <- Il fait le travail, sans plus.


# 5. Sortie structurée (label + confiance + justification)

In [16]:
from typing import Literal
from pydantic import BaseModel, Field

class Sentiment(BaseModel):
    label: Literal["positif", "négatif", "neutre"] = Field(description="Sentiment global du texte")
    confidence: float = Field(ge=0, le=1, description="Confiance entre 0 et 1")
    justification: str = Field(description="Explication courte en une phrase")

structured_llm = llm.with_structured_output(Sentiment)

for r in reviews:
    res = structured_llm.invoke([
        {"role": "system", "content": "Analyse le sentiment du texte fourni."},
        {"role": "user", "content": r},
    ])
    print(res.model_dump())

{'label': 'positif', 'confidence': 0.99, 'justification': 'Le texte exprime un enthousiasme clair et une recommandation positive.'}
{'label': 'négatif', 'confidence': 0.96, 'justification': 'Le texte mentionne un retard de livraison et un article cassé, exprimant une forte déception.'}
{'label': 'neutre', 'confidence': 0.78, 'justification': 'Le texte indique que le travail est accompli mais sans enthousiasme, ce qui correspond à un sentiment neutre.'}


# 6. Évaluation sur un petit jeu de données

In [17]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

data = [
    ("J'adore ce téléphone, la batterie tient deux jours !", "positif"),
    ("Excellent rapport qualité-prix.", "positif"),
    ("Le personnel était charmant et attentionné.", "positif"),
    ("Film magnifique, j'ai pleuré de joie.", "positif"),
    ("Horrible, il est tombé en panne au bout d'une semaine.", "négatif"),
    ("Une perte de temps et d'argent.", "négatif"),
    ("Le serveur était désagréable et la nourriture froide.", "négatif"),
    ("Application lente et pleine de bugs, je désinstalle.", "négatif"),
    ("Le colis contient un câble et un manuel.", "neutre"),
    ("La réunion est prévue à 14h en salle B.", "neutre"),
    ("Le produit est conforme à la description.", "neutre"),
    ("Il fait le travail, sans plus.", "neutre"),
]
df = pd.DataFrame(data, columns=["texte", "vrai"])

def predict(text, system):
    out = llm.invoke([{"role": "system", "content": system},
                      {"role": "user", "content": text}])
    return out.content.strip().lower().rstrip(".")

labels = ["positif", "négatif", "neutre"]
for name, sysmsg in [("zero-shot", system_zero), ("few-shot", system_few)]:
    df[name] = df["texte"].apply(lambda t: predict(t, sysmsg))
    print(f"\n=== {name} === accuracy = {accuracy_score(df['vrai'], df[name]):.2f}")
    print(classification_report(df["vrai"], df[name], labels=labels, zero_division=0))
    print(confusion_matrix(df["vrai"], df[name], labels=labels))

df


=== zero-shot === accuracy = 0.58
              precision    recall  f1-score   support

     positif       0.80      1.00      0.89         4
     négatif       0.00      0.00      0.00         4
      neutre       1.00      0.75      0.86         4

   micro avg       0.88      0.58      0.70        12
   macro avg       0.60      0.58      0.58        12
weighted avg       0.60      0.58      0.58        12

[[4 0 0]
 [0 0 0]
 [1 0 3]]

=== few-shot === accuracy = 0.58
              precision    recall  f1-score   support

     positif       0.80      1.00      0.89         4
     négatif       0.00      0.00      0.00         4
      neutre       1.00      0.75      0.86         4

   micro avg       0.88      0.58      0.70        12
   macro avg       0.60      0.58      0.58        12
weighted avg       0.60      0.58      0.58        12

[[4 0 0]
 [0 0 0]
 [1 0 3]]


,texte,vrai,zero-shot,few-shot
0,"J'adore ce téléphone, la batterie tient deux j...",positif,positif,positif
1,Excellent rapport qualité-prix.,positif,positif,positif
2,Le personnel était charmant et attentionné.,positif,positif,positif
3,"Film magnifique, j'ai pleuré de joie.",positif,positif,positif
4,"Horrible, il est tombé en panne au bout d'une ...",négatif,negatif,negatif
5,Une perte de temps et d'argent.,négatif,negatif,negatif
6,Le serveur était désagréable et la nourriture ...,négatif,negatif,negatif
7,"Application lente et pleine de bugs, je désins...",négatif,negatif,negatif
8,Le colis contient un câble et un manuel.,neutre,neutre,neutre
9,La réunion est prévue à 14h en salle B.,neutre,neutre,neutre


# 7. Autres LLM (optionnel)

## 7.1 Google Gemini (clé `GOOGLE_API_KEY`)

In [19]:
from langchain_google_genai import ChatGoogleGenerativeAI
from IPython.display import Markdown, display

llm_gemini = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)

resp3 = llm_gemini.invoke([
    {"role": "system", "content": "Vous êtes un expert en marketing digital"},
    {"role": "user",   "content": "C'est quoi un digital marketer ?"},
])

# Sur les modèles Gemini récents, .content peut être une liste de blocs et non un simple texte
text = resp3.content if isinstance(resp3.content, str) else "".join(
    b.get("text", "") if isinstance(b, dict) else str(b) for b in resp3.content
)
display(Markdown(text))

En tant qu'expert du domaine, je définirais le **digital marketer** (ou spécialiste du marketing digital) comme **l'architecte de la présence en ligne d'une marque**. 

C'est un professionnel dont la mission est d'utiliser l'ensemble des canaux numériques (Internet, réseaux sociaux, smartphones, etc.) pour **attirer des prospects, les convertir en clients et les fidéliser**.

Contrairement au marketing traditionnel (panneaux d'affichage, télévision, presse papier), le digital marketer travaille dans un univers où **tout est mesurable, testable et optimisable en temps réel**.

Voici ce qu'il faut savoir pour bien comprendre ce métier :

---

### 1. Ses missions principales : Le cycle "Attirer, Convertir, Fidéliser"

Le travail du digital marketer s'articule généralement autour de 4 grands objectifs :
1. **La notoriété (Branding) :** Faire connaître l'entreprise auprès de la bonne cible.
2. **L'acquisition de trafic :** Attirer des visiteurs qualifiés sur un site web, une application ou une page de vente.
3. **La conversion :** Transformer ces visiteurs en "leads" (prospects identifiés) ou directement en clients acheteurs.
4. **La fidélisation (Rétention) :** Maintenir le lien avec les clients pour qu'ils rachètent et deviennent des ambassadeurs de la marque.

---

### 2. Sa boîte à outils : Les grands leviers digitaux

Un digital marketer ne fait pas "juste des posts sur Instagram". Il pilote un écosystème complexe composé de plusieurs leviers :

* **Le SEO (Référencement naturel) :** Optimiser un site pour qu'il apparaisse en premier sur Google gratuitement.
* **Le SEA / Media Buying (Publicité payante) :** Créer et rentabiliser des campagnes de pub (Google Ads, Meta Ads, TikTok Ads, LinkedIn Ads).
* **Le Content Marketing (Marketing de contenu) :** Créer de la valeur via des articles de blog, des vidéos, des podcasts ou des livres blancs.
* **Le Social Media Marketing :** Développer et animer une communauté sur les réseaux sociaux.
* **L'Emailing & Marketing Automation :** Envoyer des séquences d'emails personnalisés au bon moment pour nourrir la relation client.
* **L'Analyse de données (Web Analytics) :** Utiliser des outils comme Google Analytics pour mesurer le trafic, le comportement des utilisateurs et le Retour sur Investissement (ROI).

---

### 3. "Couteau suisse" ou Ultra-spécialiste ?

Le terme "digital marketer" est un terme parapluie. Il existe deux grands types de profils :

* **Le généraliste (profil en "T") :** Souvent présent dans les PME ou startups. Il touche à tout : un peu de pub, un peu de SEO, un peu de réseaux sociaux, de la stratégie globale.
* **Le spécialiste :** Plutôt présent en agence ou dans les grands groupes. Il se concentre sur une seule discipline. Exemples : *SEO Manager, Traffic Manager (Media Buyer), CRM Manager, Growth Hacker, Copywriter*.

---

### 4. Le profil idéal d'un bon Digital Marketer

Aujourd'hui, un excellent digital marketer combine deux cerveaux :
* **Un côté analytique (Data-driven) :** Il est obsédé par les chiffres, les KPI (indicateurs de performance), le taux de conversion et la rentabilité. Il ne devine pas, il teste (A/B testing).
* **Un côté créatif et empathique :** Il doit comprendre la psychologie humaine pour concevoir des messages percutants (copywriting) et des visuels qui captent l'attention en moins de 3 secondes.
* **Une grande agilité :** Le digital évolue constamment (mises à jour des algorithmes, fin des cookies tiers, arrivée de l'Intelligence Artificielle). Il doit être en veille permanente.

**En résumé :** Le digital marketer est le moteur de croissance moderne des entreprises. Sans lui, un produit exceptionnel reste invisible dans l'immensité du web.

## 7.2 Ollama (local / cloud)

Dans PowerShell : installer Ollama puis `ollama run qwen3.5:cloud` (nécessite `ollama signin` pour les modèles `:cloud`).

In [24]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm2 = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

resp2 = llm2.invoke([
    ("system", "Vous êtes un expert en marketing digital"),
    ("human", "C'est quoi un digital marketer ?")
])

display(Markdown(resp2.text))

En tant qu'expert, je vous dirais qu'un **digital marketer** (ou marketeur digital) est le chef d'orchestre de la présence en ligne d'une entreprise. 

Son rôle principal est d'**utiliser tous les canaux numériques disponibles pour attirer des prospects, les convertir en clients, et les fidéliser.**

Contrairement au marketing traditionnel (panneaux d'affichage, radio, presse papier), le digital marketer évolue dans un univers où **tout se mesure, se teste et s'optimise en temps réel**.

Voici concrètement ce qui définit ce métier aujourd'hui :

---

### 1. Ses terrains de jeu (Les canaux)
Un digital marketer ne fait pas "juste des posts sur Instagram". Il maîtrise un écosystème complexe :
* **Le SEO (Référencement naturel) :** Optimiser un site pour qu'il apparaisse en premier sur Google gratuitement.
* **Le SEA / Media Buying (Publicité payante) :** Gérer des budgets sur Google Ads, Meta Ads (Facebook/Instagram), TikTok Ads ou LinkedIn Ads pour générer des résultats immédiats.
* **Le Content Marketing :** Créer des contenus à forte valeur ajoutée (articles de blog, vidéos, podcasts, livres blancs) pour éduquer et captiver l'audience.
* **Le Social Media Management :** Fédérer et engager une communauté sur les réseaux sociaux.
* **L’Email Marketing & Automation :** Envoyer le bon message, à la bonne personne, au bon moment grâce à des scénarios automatisés.
* **L'Analyse de données (Web Analytics) :** Utiliser des outils comme Google Analytics pour comprendre le comportement des utilisateurs et calculer le retour sur investissement (ROI).

---

### 2. Sa mission au quotidien
Le quotidien d'un digital marketer tourne autour de l'entonnoir de conversion (*Funnel marketing*) :
1. **Attirer (Trafic) :** Faire venir des visiteurs qualifiés sur un site web ou une application.
2. **Convertir (Lead Generation / Vente) :** Inciter ces visiteurs à laisser leur email, remplir un formulaire ou acheter un produit.
3. **Fidéliser (Rétention) :** Faire en sorte que les clients reviennent et recommandent la marque.
4. **Analyser et Itérer :** Regarder les chiffres, couper ce qui ne marche pas, et doubler la mise sur ce qui fonctionne (*Growth mindset*).

---

### 3. Les deux grands types de profils
Dans l'industrie, on distingue souvent :
* **Le profil "Généraliste" (ou T-Shaped) :** Souvent présent dans les PME ou les startups. Il a une vision à 360° et touche un peu à tout (stratégie globale, rédaction, petites campagnes de pub, analyse).
* **Le profil "Spécialiste" :** Plus fréquent en agence ou dans les grands groupes. Il est ultra-pointu dans un domaine précis (ex: *SEO Specialist*, *Media Buyer*, *CRM Manager*, *Growth Marketer*).

---

### 4. Les compétences clés aujourd'hui
Pour être un excellent digital marketer à l'heure actuelle, il faut combiner deux cerveaux :
* **Le côté analytique :** Aimer les chiffres, la data, le tracking et la rentabilité (ROI, CPA, CAC).
* **Le côté créatif & psychologique :** Comprendre la psychologie humaine, savoir raconter une histoire (*storytelling*) et concevoir des messages percutants (*copywriting*).
* **L'agilité technologique :** Les algorithmes changent constamment. Aujourd'hui, un bon marketer doit aussi intégrer **l'Intelligence Artificielle (comme ChatGPT, Midjourney)** dans ses processus pour être plus rapide et pertinent.

**En résumé :** Le digital marketer est le moteur de croissance moderne des entreprises. C'est le pont indispensable entre un produit/service et ses futurs clients sur Internet.

# 8. Conclusion

- Les LLM classifient sans entraînement dédié (zero-shot), et s'améliorent avec quelques exemples (few-shot).
- `temperature=0` + consigne de format strict = résultats stables et faciles à évaluer.
- La sortie structurée (Pydantic) évite le parsing manuel.
- Toujours évaluer (accuracy, matrice de confusion) sur des données étiquetées.